# 11 — Why Infeasibility Arises in GA for TSP

**Maps to:** `report/Chapters/Task2.tex` §`T2:Infeasibility`.  
**Ticket:** TICKET-11.

For each crossover operator (PMX/OX/CX), produce a worked toy example showing whether and how the permutation property breaks. Pure prose + small code demos — no new module code.

In [1]:
%run ./08_crossover.ipynb

Parent A: [0 1 2 3 4 5 6 7]
Parent B: [2 6 4 0 5 7 1 3]
Cut points: [2, 4]  →  segment: [2 3 4]
Parent A:  [0 1 2 3 4 5 6 7]
Parent B:  [2 6 4 0 5 7 1 3]
Segment:   positions [2, 4] -> [2 3 4]
Child PMX: [5 6 2 3 4 7 1 0]
Valid:     True
PMX: 1000/1000 trials produced valid permutations.
Parent A:  [0 1 2 3 4 5 6 7]
Parent B:  [2 6 4 0 5 7 1 3]
Segment:   positions [2, 4] -> [2 3 4]
Child OX:  [0 5 2 3 4 7 1 6]
Valid:     True
OX: 1000/1000 trials produced valid permutations.
Parent A:  [0 1 2 3 4 5 6 7]
Parent B:  [2 6 4 0 5 7 1 3]
Cycles:    2
Child CX:  [0 6 2 3 4 5 1 7]
Valid:     True
CX: 1000/1000 trials produced valid permutations.
Child Naive: [2 6 2 3 4 7 1 3]
Duplicates:  [np.int64(2), np.int64(3)]
Missing:     [0, 5]
Valid:       False
Naive two-point: 997/1000 trials produced INVALID permutations (99.7%)

Summary:
  PMX   — always feasible (mapping chain resolves conflicts)
  OX    — always feasible (relative-order fill skips duplicates)
  CX    — always feasible by constru

## Feasibility Requirement

For TSP, a chromosome must be a valid permutation. This means every city must appear exactly once.

If a crossover operator creates duplicate cities or missing cities, the child is infeasible.

In [7]:
parent_a = np.array([0, 1, 2, 3, 4, 5, 6, 7])
parent_b = np.array([2, 6, 4, 0, 5, 7, 1, 3])

print("Parent A:", parent_a)
print("Parent B:", parent_b)
print("Parent A valid:", is_valid_permutation(parent_a, 8))
print("Parent B valid:", is_valid_permutation(parent_b, 8))

Parent A: [0 1 2 3 4 5 6 7]
Parent B: [2 6 4 0 5 7 1 3]
Parent A valid: True
Parent B valid: True


## Naive Two-Point Crossover

A normal two-point crossover can break the permutation property because it copies a segment directly from one parent into another parent.

It does not check whether the copied cities already exist in the child.

In [3]:
def naive_two_point_crossover(parent_a, parent_b, pt1, pt2):
    child = parent_b.copy()
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    return child

child = naive_two_point_crossover(parent_a, parent_b, 2, 4)

print("Parent A:", parent_a)
print("Parent B:", parent_b)
print("Child   :", child)
print("Valid   :", is_valid_permutation(child, 8))

Parent A: [0 1 2 3 4 5 6 7]
Parent B: [2 6 4 0 5 7 1 3]
Child   : [2 6 2 3 4 7 1 3]
Valid   : False


The child is infeasible because some cities appear more than once, while some other cities are missing. Therefore, it no longer represents a valid TSP tour.

## PMX Crossover

PMX is permutation-aware. It uses mapping to resolve duplicated cities, so the child remains a valid permutation.

In [4]:
child = pmx(parent_a, parent_b, 2, 4)

print("PMX child:", child)
print("Valid    :", is_valid_permutation(child, 8))

PMX child: [5 6 2 3 4 7 1 0]
Valid    : True


## OX Crossover

OX is also permutation-aware. It copies a segment from one parent, then fills the remaining positions using the order from the other parent while skipping cities already used.

In [5]:
child = ox(parent_a, parent_b, 2, 4)

print("OX child:", child)
print("Valid   :", is_valid_permutation(child, 8))

OX child: [0 5 2 3 4 7 1 6]
Valid   : True


## CX Crossover

CX uses cycles between both parents. Each position is copied from one parent according to cycle membership, so each city appears exactly once.

In [6]:
child, cycle_count = cx(parent_a, parent_b)

print("CX child:", child)
print("Cycles  :", cycle_count)
print("Valid   :", is_valid_permutation(child, 8))

CX child: [0 6 2 3 4 5 1 7]
Cycles  : 2
Valid   : True


## Summary

| Crossover Operator | Breaks Permutation Property? | Explanation |
|---|---|---|
| Naive two-point crossover | Yes | Direct segment replacement can create duplicate and missing cities |
| PMX | No | Uses mapping to repair conflicts |
| OX | No | Skips already-used cities when filling the child |
| CX | No | Uses cycles so each city appears exactly once |

Infeasibility arises when crossover is not designed for permutation encoding. For TSP, this matters because the fitness function assumes that every city is visited exactly once. If a chromosome has duplicate or missing cities, the calculated tour length does not represent a real TSP solution.